# Gamification Affordance Mapping to SDT and CMT Outcomes and Core Study Synthesis in Wearable Fitness Technology for Higher Education Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR^2 dataset package](https://sen.science/doi/10.71728/senscience.72wn-txfe/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.72wn-txfe/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)
# Fetch (and pretty print) metadata summary
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), their `@id`s, and fields. All identifiers used come from the Croissant schema for consistent referencing.

In [ ]:
# List all record sets by their @id
print("Record Sets (@id):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')} -> {rs.get('description', '')}")

# For each record set, show their fields by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name', '(no name)')})")
    fields = rs.get('field', [])
    # A field may be a dict or a list of dicts
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  - Field @id: {field['@id']} | name: {field.get('name', field['@id'])}")

## 3. Data Extraction
Load data from all record sets into Pandas DataFrames for further analysis. We use the `@id` of each record set and field for explicit referencing.

In [ ]:
# Prepare to load all record sets into DataFrames using their @id
dataframes = dict()
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Downloading records for each record set by @id...")

for record_set_id in record_set_ids:
    # Each record is a dict with keys as field @ids
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"{record_set_id}: Loaded {len(df)} records; columns: {list(df.columns)}")
    else:
        print(f"{record_set_id}: No records loaded.")# Print first few rows and columns of the first record set as example
if record_set_ids:
    example_rs = record_set_ids[0]
    if example_rs in dataframes:
        print(f"\nExample: Columns in '{example_rs}':", dataframes[example_rs].columns.tolist())
        display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's select one record set (based on its `@id`) that contains numeric fields. We'll demonstrate filtering, normalization, and groupby using proper `@id` referencing for the fields.

In [ ]:
# Pick a record set with numeric data (adjust this @id after reviewing above outputs as needed)
# For example purposes, we'll select the first available record set and attempt to process its numeric columns.

target_record_set_id = None
numeric_field_id = None
group_field_id = None

# Find numeric fields in the record sets
for rs in dataset.record_sets:
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        data_type = field.get('dataType', '')
        if data_type in ['http://schema.org/Integer', 'http://schema.org/Float', 'http://schema.org/Number']:
            # Found a record set with numeric field
            target_record_set_id = rs['@id']
            numeric_field_id = field['@id']
            # Try to set a group field if any (choose another field not numeric)
            for gfield in fields:
                if gfield['@id'] != numeric_field_id and gfield.get('dataType','') == 'http://schema.org/Text':
                    group_field_id = gfield['@id']
                    break
            break
    if target_record_set_id:
        break

if not target_record_set_id:
    print("No suitable record set with numeric field found.")
else:
    print(f"Selected record set: {target_record_set_id}")
    print(f"Numeric field (@id): {numeric_field_id}")
    if group_field_id:
        print(f"Grouping field (@id): {group_field_id}")
    else:
        print("No suitable text field available for grouping.")

    df = dataframes[target_record_set_id]

    # Try converting the field to numeric just in case
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.9) # For demonstration, filter top 10% values
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold} (top 10%): {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalization
    mean_ = filtered_df[numeric_field_id].mean()
    std_ = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_) / std_
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by, if group field exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize the normalized numeric field distribution, and where applicable, the aggregate means by group field. This step provides a quick look at field distributions in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id and numeric_field_id and target_record_set_id in dataframes:
    filtered_numeric = filtered_df[numeric_field_id].dropna()
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_numeric, kde=True, bins=15)
    plt.title(f'Distribution of {numeric_field_id} (filtered)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        plot_data = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        sns.barplot(x=group_field_id, y=numeric_field_id, data=plot_data)
        plt.title(f'Mean {numeric_field_id} by {group_field_id} (filtered)')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No suitable numeric field to visualize.')

## 6. Conclusion
This notebook demonstrates how to load, inspect, process, and visualize tabular data defined using the FAIR^2 Croissant schema and accessed by `mlcroissant`. All steps reference record sets and fields explicitly using their `@id` fields, ensuring clarity and reproducibility. Further steps might include domain-specific analyses, machine learning, or richer visualizations tailored to the educational technology context.